# E4 — EfficientNet-B3 @ 384px DR Severity Training

**Purpose:** Train DR severity grading head on EfficientNet-B3 backbone at 384px resolution using Google Colab T4 GPU.

**Strategy:**
- Phase 1 (epochs 1–10): Freeze backbone, train severity head only
- Phase 2 (epochs 11–25): Unfreeze top 2 EfficientNet blocks + head with discriminative LR

**Target:** QWK ≥ 0.70 (from ResNet18 ceiling of 0.6456)

---

## Data Upload Instructions

Your full `Data/` folder is 96 GB, but we only need **~420 MB** (the EyePACS splits).

### Step 1: Create zip on your local machine
Already done — `eyepacs_splits.zip` (429 MB) is in your project root.

### Step 2: Upload to Google Drive
Upload `eyepacs_splits.zip` to: **My Drive/eye-realtime-inference/**

### Step 3: Run the notebook
1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells — data auto-extracts to local SSD with Windows path fix

## 0. GPU Check & Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Install dependencies
!pip install -q scikit-learn tqdm pandas pillow

## 1. Mount Google Drive & Extract Data to Local SSD

Reading images from Google Drive during training is **very slow** (~5x slower than local SSD).
This cell unzips your data to Colab's local `/content/data/` for fast I/O, and saves models back to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import zipfile
import time

# ============================================================
# CONFIGURE THIS: path to your project folder in Google Drive
# ============================================================
DRIVE_PROJECT = "/content/drive/MyDrive/eye-realtime-inference"
ZIP_PATH = os.path.join(DRIVE_PROJECT, "eyepacs_splits.zip")
LOCAL_DATA = "/content/data"  # Fast local SSD

# Verify zip exists
assert os.path.isfile(ZIP_PATH), (
    f"Zip file not found at {ZIP_PATH}\n"
    f"Upload eyepacs_splits.zip to: My Drive/eye-realtime-inference/"
)
zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"Found zip: {zip_size_mb:.0f} MB")

# Extract to local SSD with Windows backslash fix
# PowerShell's Compress-Archive uses \ separators which become literal
# characters on Linux. We fix this during extraction.
train_dir = os.path.join(LOCAL_DATA, "train")
val_dir = os.path.join(LOCAL_DATA, "val")

if not os.path.isdir(train_dir) or len(os.listdir(train_dir)) == 0:
    # Clean any previous bad extraction
    if os.path.isdir(LOCAL_DATA):
        shutil.rmtree(LOCAL_DATA)

    print("Extracting to local SSD (fixing Windows path separators)...")
    start = time.time()
    os.makedirs(LOCAL_DATA, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        for member in zf.infolist():
            # Fix Windows backslashes → forward slashes
            fixed_name = member.filename.replace('\\', '/')

            # Skip directory entries
            if fixed_name.endswith('/'):
                continue

            # Build proper output path
            out_path = os.path.join(LOCAL_DATA, fixed_name)
            os.makedirs(os.path.dirname(out_path), exist_ok=True)

            # Extract file
            with zf.open(member) as src, open(out_path, 'wb') as dst:
                shutil.copyfileobj(src, dst)

    elapsed = time.time() - start
    print(f"Extraction complete in {elapsed:.0f}s")
else:
    print("Data already extracted, skipping.")

# Auto-detect paths (zip structure depends on how it was created)
def find_dir_with_images(base, candidates):
    """Find first candidate path that exists and contains images."""
    for c in candidates:
        p = os.path.join(base, c)
        if os.path.isdir(p):
            # Check if it has images (directly or in subdirs)
            has_images = any(
                f.endswith(('.jpeg', '.jpg', '.png'))
                for f in os.listdir(p)
            )
            has_subdirs = any(os.path.isdir(os.path.join(p, d)) for d in os.listdir(p))
            if has_images or has_subdirs:
                return p
    return None

train_dir = find_dir_with_images(LOCAL_DATA, [
    "train", "eyepacs/train",
    "Data/splits/fundus/eyepacs/train",
    "splits/fundus/eyepacs/train",
])
val_dir = find_dir_with_images(LOCAL_DATA, [
    "val", "eyepacs/val",
    "Data/splits/fundus/eyepacs/val",
    "splits/fundus/eyepacs/val",
])

# Find labels CSV
labels_csv = None
for c in ["eyepacs_trainLabels.csv", "Data/labels/eyepacs_trainLabels.csv", "labels/eyepacs_trainLabels.csv"]:
    p = os.path.join(LOCAL_DATA, c)
    if os.path.isfile(p):
        labels_csv = p
        break

# Debug: show what was extracted if paths not found
if not train_dir or not val_dir or not labels_csv:
    top_items = os.listdir(LOCAL_DATA)[:20]
    print(f"DEBUG — top-level contents of {LOCAL_DATA}: {top_items}")
    if train_dir:
        print(f"DEBUG — train_dir contents: {os.listdir(train_dir)[:10]}")

assert train_dir, f"Could not find train images in {LOCAL_DATA}"
assert val_dir, f"Could not find val images in {LOCAL_DATA}"
assert labels_csv, f"Could not find labels CSV in {LOCAL_DATA}"

# Count images (including subdirectories like DR/, NORMAL/)
def count_images(directory):
    total = 0
    for root, dirs, files in os.walk(directory):
        total += sum(1 for f in files if f.endswith(('.jpeg', '.jpg', '.png')))
    return total

train_count = count_images(train_dir)
val_count = count_images(val_dir)

print(f"\n✅ Data ready on local SSD:")
print(f"  Train: {train_dir} ({train_count} images)")
print(f"  Val:   {val_dir} ({val_count} images)")
print(f"  CSV:   {labels_csv}")

## 2. Model & Data Code (Self-Contained)

All model and data loading code is included inline so the notebook runs without cloning the repo.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from collections import Counter
import pandas as pd
from PIL import Image
import numpy as np
import random
import time
import json
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix


# ==========================
# Seed
# ==========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ==========================
# EfficientNet Backbone
# ==========================
class EfficientNetBackbone(nn.Module):
    VARIANTS = {
        "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1),
        "efficientnet_b3": (models.efficientnet_b3, models.EfficientNet_B3_Weights.IMAGENET1K_V1),
    }

    def __init__(self, variant="efficientnet_b3", pretrained=True):
        super().__init__()
        factory_fn, weights_enum = self.VARIANTS[variant]
        weights = weights_enum if pretrained else None
        model = factory_fn(weights=weights)
        self.feature_dim = model.classifier[1].in_features
        model.classifier = nn.Identity()
        self.model = model
        self.variant = variant

    def forward(self, x):
        return self.model(x)

    def get_finetune_layers(self):
        features = self.model.features
        return [features[-1], features[-2]]


# ==========================
# Severity Head
# ==========================
class DRSeverityHead(nn.Module):
    def __init__(self, feature_dim, num_classes=5, dropout=0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, features):
        return self.classifier(features)


# ==========================
# Binary Head (placeholder)
# ==========================
class DRBinaryHead(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, 1),
        )

    def forward(self, features):
        return self.classifier(features)


# ==========================
# Multi-Task Model
# ==========================
class MultiTaskModel(nn.Module):
    def __init__(self, backbone="efficientnet_b3", backbone_pretrained=True):
        super().__init__()
        self.backbone = EfficientNetBackbone(variant=backbone, pretrained=backbone_pretrained)
        self.dr_binary_head = DRBinaryHead(feature_dim=self.backbone.feature_dim)
        self.dr_severity_head = DRSeverityHead(feature_dim=self.backbone.feature_dim)

    def forward(self, x):
        features = self.backbone(x)
        return {
            "dr_binary": self.dr_binary_head(features),
            "dr_severity": self.dr_severity_head(features),
        }


# ==========================
# Dataset
# ==========================
class EyePACSSeverityDataset(Dataset):
    def __init__(self, images_dir, labels_csv, transform=None):
        self.images_dir = Path(images_dir)
        self.transform = transform

        df = pd.read_csv(labels_csv)
        self.label_map = {str(row['image']): int(row['level']) for _, row in df.iterrows()}

        valid_ext = {'.jpg', '.jpeg', '.png', '.ppm', '.bmp', '.tif', '.tiff'}
        self.samples = []
        for img_path in sorted(self.images_dir.rglob('*')):
            if img_path.suffix.lower() in valid_ext:
                img_id = img_path.stem
                if img_id in self.label_map:
                    self.samples.append((img_path, self.label_map[img_id]))

        print(f"  Loaded {len(self.samples)} images from {images_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, severity = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        dr_label = 0 if severity == 0 else 1
        return image, torch.tensor(dr_label, dtype=torch.long), torch.tensor(severity, dtype=torch.long)


print("✅ All model and data classes defined.")

## 3. Configuration

In [ ]:
# ==============================
# E4 HYPERPARAMETERS
# ==============================
BACKBONE = "efficientnet_b3"
IMAGE_SIZE = 384
BATCH_SIZE = 16           # T4 16GB can handle this at 384px
NUM_EPOCHS = 25
UNFREEZE_EPOCH = 10       # Phase 2 starts here
LR_HEAD = 1e-4
LR_BACKBONE_FT = 1e-5    # 10x lower for backbone fine-tuning
WEIGHT_DECAY = 1e-4
SEED = 42
NUM_WORKERS = 2           # Colab has 2 CPU cores

# Paths — data from local SSD, saves to Google Drive (persistent)
TRAIN_DIR = train_dir     # from extraction cell
VAL_DIR = val_dir
LABELS_CSV = labels_csv
SAVE_DIR = os.path.join(DRIVE_PROJECT, "models/stage2_efficientnet")
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Backbone:   {BACKBONE}")
print(f"Image size: {IMAGE_SIZE}px")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs:     {NUM_EPOCHS} (unfreeze at {UNFREEZE_EPOCH})")
print(f"Data:       local SSD ({LOCAL_DATA})")
print(f"Save dir:   {SAVE_DIR} (Google Drive)")

## 4. Data Loading

In [ ]:
set_seed(SEED)

normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

transform_train = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    normalize,
])

transform_eval = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    normalize,
])

train_dataset = EyePACSSeverityDataset(
    images_dir=TRAIN_DIR,
    labels_csv=LABELS_CSV,
    transform=transform_train,
)

val_dataset = EyePACSSeverityDataset(
    images_dir=VAL_DIR,
    labels_csv=LABELS_CSV,
    transform=transform_eval,
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

# Class distribution
labels = [s for _, s in train_dataset.samples]
dist = Counter(labels)
print(f"\nClass distribution: {dict(sorted(dist.items()))}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 5. Model Initialization

In [ ]:
device = torch.device("cuda")

model = MultiTaskModel(backbone=BACKBONE, backbone_pretrained=True)
model.to(device)

print(f"Backbone: {BACKBONE}")
print(f"Feature dim: {model.backbone.feature_dim}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,}")

# Freeze backbone + binary head for Phase 1
for param in model.backbone.parameters():
    param.requires_grad = False
for param in model.dr_binary_head.parameters():
    param.requires_grad = False

model.backbone.eval()
model.dr_binary_head.eval()
model.dr_severity_head.train()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1 trainable: {trainable:,} / {total_params:,} ({100*trainable/total_params:.1f}%)")

## 6. Training Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, epoch_num):
    model.dr_severity_head.train()
    running_loss = 0.0
    num_batches = 0

    pbar = tqdm(loader, desc=f"Train E{epoch_num}", dynamic_ncols=True)
    for images, dr_labels, severity_labels in pbar:
        images = images.to(device)
        severity_labels = severity_labels.to(device)

        optimizer.zero_grad()
        logits = model(images)["dr_severity"]
        loss = criterion(logits, severity_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({"loss": f"{running_loss/num_batches:.4f}"})

    return running_loss / num_batches


def validate(model, loader, criterion, device):
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, dr_labels, severity_labels in tqdm(loader, desc="Val", leave=False):
            images = images.to(device)
            severity_labels = severity_labels.to(device)

            logits = model(images)["dr_severity"]
            loss = criterion(logits, severity_labels)
            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(severity_labels.cpu().numpy())

    qwk = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    return val_loss / len(loader), qwk, accuracy, all_preds, all_labels


def save_checkpoint(model, optimizer, epoch, path, **extra):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    checkpoint.update(extra)
    torch.save(checkpoint, path)


def unfreeze_top_blocks(model):
    """Unfreeze last 2 EfficientNet feature blocks."""
    blocks = model.backbone.get_finetune_layers()
    total_unfrozen = 0
    for block in blocks:
        for param in block.parameters():
            param.requires_grad = True
            total_unfrozen += param.numel()
        block.train()

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Unfrozen backbone params: {total_unfrozen:,}")
    print(f"Total trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
    return blocks


print("✅ Training functions defined.")

## 7. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()

# Phase 1 optimizer: severity head only
optimizer = torch.optim.AdamW(
    model.dr_severity_head.parameters(),
    lr=LR_HEAD,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

best_qwk = -1.0
history = []
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    # Phase transition
    if epoch == UNFREEZE_EPOCH:
        print(f"\n{'='*60}")
        print(f"Phase 2: Unfreezing top EfficientNet blocks (epoch {epoch+1})")
        print(f"{'='*60}")

        blocks = unfreeze_top_blocks(model)

        optimizer = torch.optim.AdamW([
            {"params": model.dr_severity_head.parameters(), "lr": LR_HEAD * 0.5},
            *[{"params": block.parameters(), "lr": LR_BACKBONE_FT} for block in blocks],
        ], weight_decay=WEIGHT_DECAY)

        remaining = NUM_EPOCHS - epoch
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=remaining, eta_min=1e-6
        )

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}] {'(Phase 2)' if epoch >= UNFREEZE_EPOCH else '(Phase 1)'}")

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch+1)
    val_loss, val_qwk, val_acc, _, _ = validate(model, val_loader, criterion, device)

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    epoch_time = time.time() - epoch_start

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"Val QWK: {val_qwk:.4f} | Val Acc: {val_acc:.4f} | LR: {lr:.2e} | Time: {epoch_time:.0f}s")

    history.append({
        "epoch": epoch + 1,
        "train_loss": round(train_loss, 4),
        "val_loss": round(val_loss, 4),
        "val_qwk": round(val_qwk, 4),
        "val_acc": round(val_acc, 4),
        "lr": lr,
        "time_sec": round(epoch_time, 1),
        "phase": 1 if epoch < UNFREEZE_EPOCH else 2,
    })

    if val_qwk > best_qwk:
        best_qwk = val_qwk
        save_checkpoint(
            model, optimizer, epoch,
            path=os.path.join(SAVE_DIR, "best.pt"),
            backbone=BACKBONE, image_size=IMAGE_SIZE,
            val_qwk=val_qwk, val_acc=val_acc,
        )
        print(f"✅ New best model saved (QWK={val_qwk:.4f})")

    # Save latest + history every epoch
    save_checkpoint(
        model, optimizer, epoch,
        path=os.path.join(SAVE_DIR, "latest.pt"),
        backbone=BACKBONE, image_size=IMAGE_SIZE,
        val_qwk=val_qwk, val_acc=val_acc,
    )
    with open(os.path.join(SAVE_DIR, "training_history.json"), "w") as f:
        json.dump(history, f, indent=2)

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"Training Complete! Best QWK: {best_qwk:.4f}")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Model saved to: {SAVE_DIR}")

## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]
qwks = [h['val_qwk'] for h in history]
accs = [h['val_acc'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(epochs, train_losses, 'b-o', label='Train Loss', markersize=4)
axes[0].plot(epochs, val_losses, 'r-o', label='Val Loss', markersize=4)
axes[0].axvline(x=UNFREEZE_EPOCH, color='green', linestyle='--', alpha=0.7, label='Phase 2 Start')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# QWK
axes[1].plot(epochs, qwks, 'g-o', markersize=4)
axes[1].axvline(x=UNFREEZE_EPOCH, color='green', linestyle='--', alpha=0.7, label='Phase 2 Start')
axes[1].axhline(y=0.6456, color='orange', linestyle='--', alpha=0.7, label='ResNet18 Best (0.6456)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('QWK')
axes[1].set_title('Quadratic Weighted Kappa'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Accuracy
axes[2].plot(epochs, accs, 'm-o', markersize=4)
axes[2].axvline(x=UNFREEZE_EPOCH, color='green', linestyle='--', alpha=0.7, label='Phase 2 Start')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Accuracy')
axes[2].set_title('Validation Accuracy'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Best QWK: {max(qwks):.4f} at epoch {epochs[np.argmax(qwks)]}")

## 9. Final Evaluation on Best Model

In [ ]:
# Load best checkpoint
best_path = os.path.join(SAVE_DIR, "best.pt")
checkpoint = torch.load(best_path, map_location=device, weights_only=False)

model_eval = MultiTaskModel(backbone=BACKBONE, backbone_pretrained=False)
model_eval.load_state_dict(checkpoint["model_state_dict"])
model_eval.to(device)
model_eval.eval()

print(f"Loaded best checkpoint from epoch {checkpoint['epoch'] + 1}")
print(f"Checkpoint QWK: {checkpoint.get('val_qwk', 'N/A')}")

# Run validation
_, final_qwk, final_acc, all_preds, all_labels = validate(
    model_eval, val_loader, criterion, device
)

print(f"\n{'='*50}")
print(f"FINAL EVALUATION")
print(f"{'='*50}")
print(f"QWK:      {final_qwk:.4f}")
print(f"Accuracy: {final_acc:.4f}")

# Classification report
class_names = ['No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'Proliferative (4)']
print(f"\n{classification_report(all_labels, all_preds, target_names=class_names, zero_division=0)}")

# Confusion matrix
import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'E4 EfficientNet-B3 Confusion Matrix (QWK={final_qwk:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Comparison with ResNet18 Baseline

| Metric | ResNet18 @224px (E3+) | EfficientNet-B3 @384px (E4) |
|--------|----------------------|----------------------------|
| QWK    | 0.6456               | *see above*                |
| Accuracy | 0.7928             | *see above*                |
| Mild Recall | 0.00%           | *see above*                |
| Params | 11.2M               | 12.3M                      |
| Resolution | 224px            | 384px                      |

In [ ]:
# ==========================================
# E4 FORENSIC ANALYSIS — Mild DR Breakdown
# ==========================================
# This is the critical diagnostic before deciding E4b

# Per-class recall
print("Per-Class Recall:")
for i, name in enumerate(class_names):
    mask = np.array(all_labels) == i
    if mask.sum() > 0:
        acc = (np.array(all_preds)[mask] == i).mean()
        marker = " ← CRITICAL" if i == 1 else ""
        print(f"  {name}: {acc:.1%} ({mask.sum()} samples){marker}")

# Mild DR forensic: WHERE do class 1 predictions go?
mild_mask = np.array(all_labels) == 1
mild_preds = np.array(all_preds)[mild_mask]
n_mild = mild_mask.sum()
print(f"\n{'='*50}")
print(f"MILD DR FORENSIC (n={n_mild})")
print(f"{'='*50}")
print("Where Mild (class 1) samples get predicted:")
for j, name in enumerate(class_names):
    count = (mild_preds == j).sum()
    pct = count / n_mild * 100 if n_mild > 0 else 0
    arrow = "←←← COLLAPSE" if j == 0 and pct > 80 else ("← ordinal neighbor" if j == 2 else "")
    print(f"  → {name}: {count:>4} ({pct:>5.1f}%) {arrow}")

# Check ordinal smoothness: are errors near-class or far-class?
print(f"\nError Distance Analysis:")
correct = (np.array(all_preds) == np.array(all_labels))
wrong_mask = ~correct
wrong_preds = np.array(all_preds)[wrong_mask]
wrong_labels = np.array(all_labels)[wrong_mask]
distances = np.abs(wrong_preds - wrong_labels)
print(f"  Total errors: {wrong_mask.sum()} / {len(all_labels)} ({wrong_mask.mean():.1%})")
print(f"  Mean error distance: {distances.mean():.2f}")
print(f"  ±1 class errors: {(distances == 1).sum()} ({(distances == 1).mean():.1%} of errors)")
print(f"  ±2+ class errors: {(distances >= 2).sum()} ({(distances >= 2).mean():.1%} of errors)")

# Decision gate
mild_recall = (mild_preds == 1).sum() / n_mild if n_mild > 0 else 0
mild_to_zero = (mild_preds == 0).sum() / n_mild if n_mild > 0 else 0
print(f"\n{'='*50}")
print(f"DECISION GATE")
print(f"{'='*50}")
print(f"Mild recall: {mild_recall:.1%}")
print(f"Mild→0 collapse: {mild_to_zero:.1%}")
if mild_recall < 0.05:
    print(f"\n⚠️  Mild recall ≈ 0% — confirms loss-objective misalignment.")
    print(f"   → Proceed with E4b (EMD loss) as 5-epoch diagnostic.")
else:
    print(f"\n✅ Mild recall > 5% — architecture helped.")
    print(f"   → Consider whether E4b (EMD) is still needed.")

## 11. Download Model to Local Machine

The best model is already saved to Google Drive at the configured path. You can also download it directly:

In [ ]:
# Download best model to local machine
from google.colab import files
files.download(os.path.join(SAVE_DIR, "best.pt"))

## 12. E4b — EMD Loss + Full Backbone Unfreeze (5-Epoch Diagnostic)

**Core insight:** The bottleneck is NOT representation capacity — it's loss-objective misalignment.

| What we optimize | What we measure |
|---|---|
| CrossEntropy (per-class log-likelihood) | QWK (ordinal distance²) |
| Treats 0→1 same as 0→4 | Penalizes proportional to distance² |
| Rewards majority class confidence | Rewards ordinal boundary calibration |

**Solution: Earth Mover's Distance (EMD) Loss**
- Minimizes distance between predicted CDF and true CDF
- Directly penalizes ordinal mispredictions proportional to their distance
- Proven in DR grading literature (Google/Verily retinal screening)

**Strategy — GPU-efficient:**
1. Run **5 epochs only** as diagnostic (~30 min on T4)
2. Check: Mild recall, confusion matrix, error distances
3. **If Mild > 0%** → continue to 15 more epochs (cell 13)
4. **If Mild = 0%** → STOP, pivot to label noise / boundary audit

In [ ]:
# =============================================
# 🔬 GRADIENT SANITY CHECK — Run BEFORE training
# =============================================
# EMD implementations can silently detach tensors.
# This cell verifies gradients flow properly in a single forward+backward pass.

class EMDLoss(nn.Module):
    """Squared Earth Mover's Distance for ordinal classification."""
    def __init__(self, num_classes=5):
        super().__init__()
        self.num_classes = num_classes

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)               # (B, C) — differentiable
        cdf_pred = torch.cumsum(probs, dim=1)               # (B, C) — differentiable
        one_hot = torch.zeros_like(logits)                   # constant w.r.t. params
        one_hot.scatter_(1, targets.unsqueeze(1), 1.0)
        cdf_true = torch.cumsum(one_hot, dim=1)             # constant (target CDF)
        emd = torch.mean(torch.sum((cdf_pred - cdf_true) ** 2, dim=1))
        return emd

# Quick test: 1 batch, forward + backward, check gradients
test_model = MultiTaskModel(backbone=BACKBONE, backbone_pretrained=False)
test_model.to(device)
test_model.train()

# Enable grad for all params temporarily
for p in test_model.parameters():
    p.requires_grad = True

test_criterion = EMDLoss(num_classes=5)
test_batch = next(iter(val_loader))
test_images = test_batch[0][:4].to(device)  # just 4 images
test_labels = test_batch[2][:4].to(device)

# Forward
test_logits = test_model(test_images)["dr_severity"]
test_loss = test_criterion(test_logits, test_labels)

# Verify loss properties
print(f"EMD loss value:    {test_loss.item():.4f}")
print(f"Loss requires_grad: {test_loss.requires_grad}")
print(f"Loss grad_fn:       {test_loss.grad_fn}")
assert test_loss.requires_grad, "❌ FATAL: Loss has no gradient! Check EMD implementation."
assert test_loss.grad_fn is not None, "❌ FATAL: Loss has no grad_fn! Tensor was detached."

# Backward
test_loss.backward()

# Check gradients exist and are non-zero
head_params = list(test_model.dr_severity_head.parameters())
backbone_params = list(test_model.backbone.parameters())

head_grad_norm = sum(p.grad.norm().item() for p in head_params if p.grad is not None)
backbone_grad_norm = sum(p.grad.norm().item() for p in backbone_params if p.grad is not None)
head_has_grad = sum(1 for p in head_params if p.grad is not None and p.grad.norm() > 0)
backbone_has_grad = sum(1 for p in backbone_params if p.grad is not None and p.grad.norm() > 0)

print(f"\nSeverity head:  {head_has_grad}/{len(head_params)} params have non-zero grad (norm={head_grad_norm:.6f})")
print(f"Backbone:       {backbone_has_grad}/{len(backbone_params)} params have non-zero grad (norm={backbone_grad_norm:.6f})")

assert head_grad_norm > 0, "❌ FATAL: Head has zero gradients!"
assert backbone_grad_norm > 0, "❌ FATAL: Backbone has zero gradients!"

# Verify ordinal structure: class-distance should affect loss
with torch.no_grad():
    fake_logits = torch.tensor([[10.0, 0, 0, 0, 0]], device=device)  # confident class 0
    loss_close = test_criterion(fake_logits, torch.tensor([1], device=device))  # true=1, distance=1
    loss_far = test_criterion(fake_logits, torch.tensor([4], device=device))    # true=4, distance=4
    print(f"\nOrdinal check: predict=0, true=1 → loss={loss_close.item():.4f}")
    print(f"Ordinal check: predict=0, true=4 → loss={loss_far.item():.4f}")
    assert loss_far > loss_close, "❌ FATAL: EMD doesn't penalize larger ordinal distance more!"
    print(f"Ratio (should be >> 1): {loss_far.item() / loss_close.item():.1f}x")

print(f"\n✅ ALL CHECKS PASSED — EMD gradients flow correctly, ordinal structure verified.")
print(f"   Safe to proceed with E4b training.")

# Cleanup
del test_model, test_criterion, test_logits, test_loss, test_batch
torch.cuda.empty_cache()

In [ ]:
# ==============================
# E4b CONFIG — 5-EPOCH DIAGNOSTIC
# ==============================
# EMDLoss is already defined in the gradient check cell above

DIAGNOSTIC_EPOCHS = 5       # just enough to see if EMD changes the landscape
E4B_LR_HEAD = 3e-5
E4B_LR_TOP = 5e-6           # top 2 blocks
E4B_LR_LOWER = 1e-6         # lower blocks
E4B_WEIGHT_DECAY = 1e-4

# Load E4 best checkpoint
# If you uploaded best.pt to a custom location, set the path here:
E4_CHECKPOINT_PATH = "/content/drive/MyDrive/New folder/best.pt"
best_path = E4_CHECKPOINT_PATH
checkpoint = torch.load(best_path, map_location=device, weights_only=False)

model_b = MultiTaskModel(backbone=BACKBONE, backbone_pretrained=False)
model_b.load_state_dict(checkpoint["model_state_dict"])
model_b.to(device)

e4_best_qwk = checkpoint.get("val_qwk", 0.5919)
e4_best_epoch = checkpoint.get("epoch", 19) + 1
print(f"Loaded E4 best checkpoint: epoch {e4_best_epoch}, QWK={e4_best_qwk:.4f}")

# Freeze binary head (not used)
for param in model_b.dr_binary_head.parameters():
    param.requires_grad = False
model_b.dr_binary_head.eval()

# Unfreeze EVERYTHING in backbone + severity head
for param in model_b.backbone.parameters():
    param.requires_grad = True
for param in model_b.dr_severity_head.parameters():
    param.requires_grad = True

# 3-tier discriminative LR
features = model_b.backbone.model.features
top_block_params = list(features[-1].parameters()) + list(features[-2].parameters())
top_block_ids = {id(p) for p in top_block_params}

lower_block_params = [
    p for p in model_b.backbone.parameters()
    if p.requires_grad and id(p) not in top_block_ids
]

# Use T_max=20 (full budget) so LR schedule is smooth if we continue later
optimizer_b = torch.optim.AdamW([
    {"params": model_b.dr_severity_head.parameters(), "lr": E4B_LR_HEAD},
    {"params": top_block_params, "lr": E4B_LR_TOP},
    {"params": lower_block_params, "lr": E4B_LR_LOWER},
], weight_decay=E4B_WEIGHT_DECAY)

scheduler_b = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_b, T_max=20, eta_min=1e-7  # full 20-epoch curve, stop at 5
)

# Parameter summary
total_params = sum(p.numel() for p in model_b.parameters())
trainable_params = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
print(f"\nTotal params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
print(f"Head LR: {E4B_LR_HEAD:.0e} | Top LR: {E4B_LR_TOP:.0e} | Lower LR: {E4B_LR_LOWER:.0e}")

# EMD loss (the key change — verified in gradient check cell)
criterion_b = EMDLoss(num_classes=5)
print(f"Loss function: EMD (Earth Mover's Distance)")
print(f"\n🔬 DIAGNOSTIC RUN: {DIAGNOSTIC_EPOCHS} epochs only (~30 min)")
print(f"   Looking for: Mild recall > 0%")
print("=" * 60)

# Save E4b checkpoints separately
SAVE_DIR_B = os.path.join(DRIVE_PROJECT, "models/stage2_efficientnet_e4b")
os.makedirs(SAVE_DIR_B, exist_ok=True)

best_qwk_b = -1.0
history_b = []

start_time_b = time.time()

for epoch in range(DIAGNOSTIC_EPOCHS):
    epoch_start = time.time()

    model_b.backbone.train()
    model_b.dr_severity_head.train()
    model_b.dr_binary_head.eval()

    print(f"\nEpoch [{epoch+1}/{DIAGNOSTIC_EPOCHS}] (E4b — EMD diagnostic)")

    train_loss = train_one_epoch(model_b, train_loader, optimizer_b, criterion_b, device, epoch+1)
    val_loss, val_qwk, val_acc, epoch_preds, epoch_labels = validate(
        model_b, val_loader, criterion_b, device
    )

    scheduler_b.step()
    lr = optimizer_b.param_groups[0]['lr']
    epoch_time = time.time() - epoch_start

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} (EMD)")
    print(f"Val QWK: {val_qwk:.4f} | Val Acc: {val_acc:.4f} | LR: {lr:.2e} | Time: {epoch_time:.0f}s")

    history_b.append({
        "epoch": epoch + 1,
        "global_epoch": e4_best_epoch + epoch + 1,
        "train_loss": round(train_loss, 4),
        "val_loss": round(val_loss, 4),
        "val_qwk": round(val_qwk, 4),
        "val_acc": round(val_acc, 4),
        "lr": lr,
        "time_sec": round(epoch_time, 1),
        "loss_fn": "EMD",
    })

    if val_qwk > best_qwk_b:
        best_qwk_b = val_qwk
        save_checkpoint(
            model_b, optimizer_b, epoch,
            path=os.path.join(SAVE_DIR_B, "best.pt"),
            backbone=BACKBONE, image_size=IMAGE_SIZE,
            val_qwk=val_qwk, val_acc=val_acc,
            loss_fn="EMD",
            base_checkpoint=f"E4 epoch {e4_best_epoch}",
        )
        print(f"✅ New best model saved (QWK={val_qwk:.4f})")

    save_checkpoint(
        model_b, optimizer_b, epoch,
        path=os.path.join(SAVE_DIR_B, "latest.pt"),
        backbone=BACKBONE, image_size=IMAGE_SIZE,
        val_qwk=val_qwk, val_acc=val_acc,
    )
    with open(os.path.join(SAVE_DIR_B, "training_history.json"), "w") as f:
        json.dump(history_b, f, indent=2)

total_time_b = time.time() - start_time_b
print(f"\n{'='*60}")
print(f"E4b Diagnostic Complete! ({DIAGNOSTIC_EPOCHS} epochs, {total_time_b/60:.1f} min)")
print(f"{'='*60}")
print(f"E4 best QWK (CE):   {e4_best_qwk:.4f}")
print(f"E4b best QWK (EMD): {best_qwk_b:.4f}")
print(f"Delta:              {best_qwk_b - e4_best_qwk:+.4f}")

# =============================================
# 🔬 DIAGNOSTIC: Mild DR analysis after 5 epochs
# =============================================
best_path_diag = os.path.join(SAVE_DIR_B, "best.pt")
ckpt_diag = torch.load(best_path_diag, map_location=device, weights_only=False)
model_diag = MultiTaskModel(backbone=BACKBONE, backbone_pretrained=False)
model_diag.load_state_dict(ckpt_diag["model_state_dict"])
model_diag.to(device)
model_diag.eval()

_, diag_qwk, diag_acc, diag_preds, diag_labels = validate(
    model_diag, val_loader, criterion_b, device
)

class_names = ['No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'Proliferative (4)']

# Mild forensic
mild_mask = np.array(diag_labels) == 1
mild_preds_arr = np.array(diag_preds)[mild_mask]
n_mild = mild_mask.sum()
mild_recall = (mild_preds_arr == 1).sum() / n_mild if n_mild > 0 else 0.0

print(f"\n{'='*60}")
print(f"🔬 MILD DR DIAGNOSTIC (n={n_mild})")
print(f"{'='*60}")
print(f"Mild recall: {mild_recall:.1%}")
print(f"\nWhere Mild samples get predicted:")
for j, name in enumerate(class_names):
    count = (mild_preds_arr == j).sum()
    pct = count / n_mild * 100 if n_mild > 0 else 0
    print(f"  → {name}: {count:>4} ({pct:>5.1f}%)")

# Error distance analysis
wrong_mask = np.array(diag_preds) != np.array(diag_labels)
if wrong_mask.sum() > 0:
    wrong_preds = np.array(diag_preds)[wrong_mask]
    wrong_labels = np.array(diag_labels)[wrong_mask]
    distances = np.abs(wrong_preds - wrong_labels)
    print(f"\nError Distance: mean={distances.mean():.2f}, "
          f"±1={100*(distances==1).mean():.0f}%, "
          f"±2+={100*(distances>=2).mean():.0f}%")

# Confusion matrix
import seaborn as sns
cm_diag = confusion_matrix(diag_labels, diag_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_diag, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'E4b 5-Epoch Diagnostic — Confusion Matrix (QWK={diag_qwk:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR_B, 'e4b_diagnostic_cm.png'), dpi=150, bbox_inches='tight')
plt.show()

# DECISION GATE
print(f"\n{'='*60}")
print(f"🚦 DECISION GATE")
print(f"{'='*60}")
mild_to_zero = (mild_preds_arr == 0).sum() / n_mild if n_mild > 0 else 0
mild_to_two = (mild_preds_arr == 2).sum() / n_mild if n_mild > 0 else 0

if mild_recall >= 0.05:
    print(f"✅ Mild recall = {mild_recall:.1%} — EMD IS WORKING!")
    print(f"   → Run next cell to continue training for 15 more epochs.")
    print(f"   → First real signal of class 1 detection in the entire project.")
elif mild_to_two > 0.05:
    print(f"⚡ Mild recall = {mild_recall:.1%}, but Mild→Moderate = {mild_to_two:.1%}")
    print(f"   → EMD is shifting predictions toward ordinal center (weak signal).")
    print(f"   → Run next cell — the gradient direction is correct, needs more epochs.")
else:
    print(f"❌ Mild recall = {mild_recall:.1%}, Mild→0 = {mild_to_zero:.1%}")
    print(f"   → EMD did NOT fix the collapse after {DIAGNOSTIC_EPOCHS} epochs.")
    print(f"   → Problem is likely dataset/label noise, not loss formulation.")
    print(f"   → STOP. Pivot to label audit or 0-vs-1 boundary analysis.")

## 13. E4b Continuation — Epochs 6–20 (Only If Diagnostic Passed)

**Run this ONLY if the decision gate above showed:**
- Mild recall > 5%, OR
- Mild to Moderate shifted (weak signal worth pursuing)

**DO NOT run if the gate showed stop** — pivot to label audit instead.

This resumes from the `latest.pt` checkpoint (end of epoch 5), trains 15 more epochs with the same EMD loss, optimizer state, and LR schedule.

In [ ]:
# ==============================
# E4b CONTINUATION: Epochs 6–20
# ==============================
# Resume from where diagnostic left off

CONTINUATION_EPOCHS = 15  # 6 through 20
RESUME_EPOCH = DIAGNOSTIC_EPOCHS  # should be 5

# model_b, optimizer_b, scheduler_b, criterion_b are still in memory from diagnostic
# If you restarted runtime, uncomment the block below to reload:
# ---
# latest_path = os.path.join(SAVE_DIR_B, "latest.pt")
# ckpt_resume = torch.load(latest_path, map_location=device, weights_only=False)
# model_b = MultiTaskModel(backbone=BACKBONE, backbone_pretrained=False)
# model_b.load_state_dict(ckpt_resume["model_state_dict"])
# model_b.to(device)
# criterion_b = EMDLoss(num_classes=5)
# # Re-create optimizer and scheduler, then load states
# features = model_b.backbone.model.features
# top_block_params = list(features[-1].parameters()) + list(features[-2].parameters())
# top_block_ids = {id(p) for p in top_block_params}
# lower_block_params = [p for p in model_b.backbone.parameters() if p.requires_grad and id(p) not in top_block_ids]
# optimizer_b = torch.optim.AdamW([
#     {"params": model_b.dr_severity_head.parameters(), "lr": E4B_LR_HEAD},
#     {"params": top_block_params, "lr": E4B_LR_TOP},
#     {"params": lower_block_params, "lr": E4B_LR_LOWER},
# ], weight_decay=E4B_WEIGHT_DECAY)
# optimizer_b.load_state_dict(ckpt_resume["optimizer_state_dict"])
# scheduler_b = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_b, T_max=20, eta_min=1e-7)
# for _ in range(RESUME_EPOCH): scheduler_b.step()  # advance scheduler to correct position
# best_qwk_b = max(h['val_qwk'] for h in history_b) if history_b else -1.0
# ---

print(f"Continuing E4b from epoch {RESUME_EPOCH+1} to {RESUME_EPOCH + CONTINUATION_EPOCHS}")
print(f"Current best QWK: {best_qwk_b:.4f}")
print(f"Loss: EMD | Optimizer + scheduler state preserved")
print("=" * 60)

start_time_cont = time.time()

for epoch in range(RESUME_EPOCH, RESUME_EPOCH + CONTINUATION_EPOCHS):
    epoch_start = time.time()

    model_b.backbone.train()
    model_b.dr_severity_head.train()
    model_b.dr_binary_head.eval()

    print(f"\nEpoch [{epoch+1}/20] (E4b — EMD continuation)")

    train_loss = train_one_epoch(model_b, train_loader, optimizer_b, criterion_b, device, epoch+1)
    val_loss, val_qwk, val_acc, _, _ = validate(model_b, val_loader, criterion_b, device)

    scheduler_b.step()
    lr = optimizer_b.param_groups[0]['lr']
    epoch_time = time.time() - epoch_start

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} (EMD)")
    print(f"Val QWK: {val_qwk:.4f} | Val Acc: {val_acc:.4f} | LR: {lr:.2e} | Time: {epoch_time:.0f}s")

    history_b.append({
        "epoch": epoch + 1,
        "global_epoch": e4_best_epoch + epoch + 1,
        "train_loss": round(train_loss, 4),
        "val_loss": round(val_loss, 4),
        "val_qwk": round(val_qwk, 4),
        "val_acc": round(val_acc, 4),
        "lr": lr,
        "time_sec": round(epoch_time, 1),
        "loss_fn": "EMD",
    })

    if val_qwk > best_qwk_b:
        best_qwk_b = val_qwk
        save_checkpoint(
            model_b, optimizer_b, epoch,
            path=os.path.join(SAVE_DIR_B, "best.pt"),
            backbone=BACKBONE, image_size=IMAGE_SIZE,
            val_qwk=val_qwk, val_acc=val_acc,
            loss_fn="EMD",
            base_checkpoint=f"E4 epoch {e4_best_epoch}",
        )
        print(f"✅ New best model saved (QWK={val_qwk:.4f})")

    save_checkpoint(
        model_b, optimizer_b, epoch,
        path=os.path.join(SAVE_DIR_B, "latest.pt"),
        backbone=BACKBONE, image_size=IMAGE_SIZE,
        val_qwk=val_qwk, val_acc=val_acc,
    )
    with open(os.path.join(SAVE_DIR_B, "training_history.json"), "w") as f:
        json.dump(history_b, f, indent=2)

total_time_cont = time.time() - start_time_cont
total_time_all = total_time_b + total_time_cont
print(f"\n{'='*60}")
print(f"E4b Full Training Complete! (20 epochs, {total_time_all/60:.1f} min total)")
print(f"{'='*60}")
print(f"E4 best QWK (CE):       {e4_best_qwk:.4f}")
print(f"E4b best QWK (EMD):     {best_qwk_b:.4f}")
print(f"Delta:                  {best_qwk_b - e4_best_qwk:+.4f}")
print(f"ResNet18 E3+ reference: 0.6456")

In [ ]:
# E4b Training Curves (full run)
import matplotlib.pyplot as plt

epochs_b = [h['epoch'] for h in history_b]
train_losses_b = [h['train_loss'] for h in history_b]
val_losses_b = [h['val_loss'] for h in history_b]
qwks_b = [h['val_qwk'] for h in history_b]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_b, train_losses_b, 'b-o', label='Train Loss', markersize=4)
axes[0].plot(epochs_b, val_losses_b, 'r-o', label='Val Loss', markersize=4)
axes[0].axvline(x=5.5, color='purple', linestyle=':', alpha=0.7, label='Diagnostic checkpoint')
axes[0].set_xlabel('E4b Epoch'); axes[0].set_ylabel('EMD Loss')
axes[0].set_title('E4b Loss Curves'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_b, qwks_b, 'g-o', markersize=4, label='E4b QWK')
axes[1].axhline(y=0.5919, color='blue', linestyle='--', alpha=0.7, label='E4 Best (0.5919)')
axes[1].axhline(y=0.6456, color='orange', linestyle='--', alpha=0.7, label='ResNet18 E3+ (0.6456)')
axes[1].axvline(x=5.5, color='purple', linestyle=':', alpha=0.7, label='Diagnostic checkpoint')
axes[1].set_xlabel('E4b Epoch'); axes[1].set_ylabel('QWK')
axes[1].set_title('E4b Quadratic Weighted Kappa'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR_B, 'e4b_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"E4b Best QWK: {max(qwks_b):.4f} at epoch {epochs_b[np.argmax(qwks_b)]}")

In [ ]:
# E4b FINAL Evaluation (after full 20 epochs)
best_path_b = os.path.join(SAVE_DIR_B, "best.pt")
ckpt_b = torch.load(best_path_b, map_location=device, weights_only=False)

model_eval_b = MultiTaskModel(backbone=BACKBONE, backbone_pretrained=False)
model_eval_b.load_state_dict(ckpt_b["model_state_dict"])
model_eval_b.to(device)
model_eval_b.eval()

_, final_qwk_b, final_acc_b, preds_b, labels_b = validate(
    model_eval_b, val_loader, criterion_b, device
)

class_names = ['No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'Proliferative (4)']
print(f"{'='*50}")
print(f"E4b FINAL EVALUATION (EMD Loss, 20 epochs)")
print(f"{'='*50}")
print(f"QWK:      {final_qwk_b:.4f}")
print(f"Accuracy: {final_acc_b:.4f}")
print(f"\n{classification_report(labels_b, preds_b, target_names=class_names, zero_division=0)}")

# Confusion matrix
import seaborn as sns
cm_b = confusion_matrix(labels_b, preds_b)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_b, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'E4b EMD Loss Final — Confusion Matrix (QWK={final_qwk_b:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR_B, 'e4b_final_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# Per-class accuracy
print("\nPer-Class Recall:")
for i, name in enumerate(class_names):
    mask = np.array(labels_b) == i
    if mask.sum() > 0:
        acc = (np.array(preds_b)[mask] == i).mean()
        marker = " ← CRITICAL" if i == 1 else ""
        print(f"  {name}: {acc:.1%} ({mask.sum()} samples){marker}")

# Mild forensic (final)
mild_mask = np.array(labels_b) == 1
mild_preds_final = np.array(preds_b)[mild_mask]
n_mild = mild_mask.sum()
mild_recall_final = (mild_preds_final == 1).sum() / n_mild if n_mild > 0 else 0.0

print(f"\nMild DR distribution after 20 epochs:")
for j, name in enumerate(class_names):
    count = (mild_preds_final == j).sum()
    pct = count / n_mild * 100 if n_mild > 0 else 0
    print(f"  → {name}: {count:>4} ({pct:>5.1f}%)")

# Full comparison table
print(f"\n{'='*62}")
print(f"FULL EXPERIMENT COMPARISON")
print(f"{'='*62}")
print(f"{'Experiment':<30} {'Loss':<8} {'QWK':>8} {'Acc':>8} {'Mild':>8}")
print(f"{'-'*62}")
print(f"{'E0 ResNet18 baseline':<30} {'CE':<8} {'0.6269':>8} {'0.7121':>8} {'0.0%':>8}")
print(f"{'E3+ ResNet18 layer4':<30} {'CE':<8} {'0.6456':>8} {'0.7928':>8} {'0.0%':>8}")
print(f"{'E4 EffNet-B3 384px':<30} {'CE':<8} {'0.5919':>8} {'0.7782':>8} {'0.0%':>8}")
print(f"{'E4b EffNet-B3 EMD':<30} {'EMD':<8} {final_qwk_b:>8.4f} {final_acc_b:>8.4f} {mild_recall_final:>7.1%}")